In [10]:
import pandas as pd
from pyprojroot import here

In [11]:
RAW_DATA_DIR = here() / "data" / "raw"
INTERIM_DATA_DIR = here() / "data" / "interim"

In [12]:
def norm(df):
    df['CVEGEO'] = df['CVEGEO'].astype(str).str.zfill(5) # We ensure all merging keys share the same format
    return df

In [13]:
econo       = norm(pd.read_csv(INTERIM_DATA_DIR/'economic_output.csv'))
industry    = norm(pd.read_csv(INTERIM_DATA_DIR/'industrial_density.csv'))
ciit        = norm(pd.read_csv(INTERIM_DATA_DIR/'distance_to_ciit.csv'))
border      = norm(pd.read_csv(INTERIM_DATA_DIR/'min_border_distance.csv'))
ports       = norm(pd.read_csv(INTERIM_DATA_DIR/'ports_dist_by_municipality.csv'))
highways    = norm(pd.read_csv(INTERIM_DATA_DIR/'highways_by_municipality.csv'))
energy      = norm(pd.read_csv(INTERIM_DATA_DIR/'min_dist_energy.csv'))
rails       = norm(pd.read_csv(INTERIM_DATA_DIR/'rails_by_municipality.csv'))
photo       = norm(pd.read_csv(INTERIM_DATA_DIR/'pv_potential_by_municipality.csv'))
water       = norm(pd.read_csv(INTERIM_DATA_DIR/'water_availability_by_municipality.csv'))
wind        = norm(pd.read_csv(INTERIM_DATA_DIR/'wind_potential_by_municipality.csv'))
rain        = norm(pd.read_csv(INTERIM_DATA_DIR/'rainfall_by_municipality.csv'))
crime       = norm(pd.read_csv(INTERIM_DATA_DIR/'crime_by_municipality.csv'))

In [14]:
dfs = [water, econo, industry, ciit, border, ports, highways, energy, rails, photo, wind, rain, crime]

In [15]:
merged = dfs[0]
for df in dfs[1:]:
    merged = merged.merge(df, on='CVEGEO', how='left')

In [16]:
merged.drop(columns=['Unnamed: 0'], inplace=True)
print(merged.shape)
print(merged.columns.tolist())
merged.to_csv(INTERIM_DATA_DIR/'master_merged.csv', index=False)

(2478, 15)
['CVEGEO', 'water_availability', 'economic_output', 'n_industy', 'distance_ciit', 'dist_to_border_km', 'dist_min_ports', 'highways', 'dist_min_km', 'gen_min_km', 'rails', 'pv_mean_annual_kwh', 'wind_mean', 'rainfall', 'crime']


### Descriptive summary

In [17]:
num_df = merged.select_dtypes(include='number')

summary = pd.DataFrame({
    'min':      num_df.min(),
    'max':      num_df.max(),
    'mean':     num_df.mean(),
    'median':   num_df.median(),
    'std':      num_df.std(),
    'skewness': num_df.skew(),
    'kurtosis': num_df.kurt(),
    'missing':  num_df.isna().sum(),
})

In [18]:
print(summary.to_string())
summary.to_csv(INTERIM_DATA_DIR/"master_summary.csv", encoding="utf-8")

                            min            max         mean       median           std  skewness   kurtosis  missing
water_availability     0.000000    2059.634923   131.962515     6.509252    444.335604  3.944277  14.073340        0
economic_output        0.000000  479117.157000  5724.389882    38.794000  30197.404344  8.349793  83.164438       27
n_industy              0.000000   12557.000000   264.978612    69.000000    662.839850  7.670560  89.727201        0
distance_ciit          6.414214    3829.086071   673.691161   524.472655    508.647630  1.639562   3.551463        0
dist_to_border_km      4.242641    1270.751442   612.850437   667.355952    220.729364 -0.852942   0.158919        0
dist_min_ports         3.483483     814.480541   217.539679   207.378654    109.609702  1.072154   2.898759        0
highways               0.000000       1.000000     0.964487     1.000000      0.185109 -5.022594  23.245208        0
dist_min_km            0.033195     271.640524    45.092298    3